# Bookstore Database & Business Analysis

## Project Overview

This project focuses on designing, implementing, and analysing a relational database for a fictional bookstore.

The database is built using SQLite and Python. SQL queries are used to explore relationships, validate data integrity, and analyse bookstore operations.

## Project Objectives

- Design a relational database with properly defined relationships.
- Implement primary keys, foreign keys, and integrity constraints.
- Populate the database with sample bookstore data.
- Perform SQL queries using filtering, joins, and aggregations.
- Analyse sales, revenue, customer behaviour, and inventory.
- Extract meaningful business insights from the data.

## Technologies Used

- Python
- SQLite
- Pandas
- Jupyter Notebook

#### SQLite3 connection to DATABASE:

In [58]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('bookstore.db')
cursor = conn.cursor()
print("Successfully connected to database")
conn.execute("PRAGMA foreign_keys = ON")
print(conn.execute("PRAGMA foreign_keys;").fetchone())

Successfully connected to database
(1,)


In [59]:
def run_query(query):
    return pd.read_sql_query(query, conn)
print("Result: run_query function succesfully created!")

Result: run_query function succesfully created!


#### Add tables to the DATABASE

In [ ]:
# Corrected multi-statement SQL script
create_tables = '''
CREATE TABLE IF NOT EXISTS Authors(
    author_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS Books(
    book_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    publication_year INTEGER,
    price_paise INTEGER CHECK(price_paise >=0) NOT NULL,
    stock_quantity INTEGER CHECK (stock_quantity >= 0) NOT NULL,
    isbn TEXT UNIQUE
);

CREATE TABLE IF NOT EXISTS Customers(
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE,
    phone TEXT
);

CREATE TABLE IF NOT EXISTS Orders(
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date TEXT DEFAULT (strftime('%Y-%m-%dT%H:%M:%SZ', 'now')),
    status TEXT CHECK (status IN ('pending', 'completed', 'cancelled')),
    FOREIGN KEY(customer_id) REFERENCES Customers(customer_id)
);

CREATE TABLE IF NOT EXISTS OrderItems(
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    book_id INTEGER NOT NULL,
    quantity INTEGER CHECK(quantity > 0) NOT NULL,
    unit_price INTEGER CHECK(unit_price >=0) NOT NULL,
    FOREIGN KEY(order_id) REFERENCES Orders(order_id),
    FOREIGN KEY(book_id) REFERENCES Books(book_id)
);

CREATE TABLE IF NOT EXISTS BookAuthors(
    author_id INTEGER NOT NULL,
    book_id INTEGER NOT NULL,
    PRIMARY KEY(author_id, book_id),
    FOREIGN KEY(author_id) REFERENCES Authors(author_id),
    FOREIGN KEY(book_id) REFERENCES Books(book_id)
);
'''

cursor.executescript(create_tables)
conn.commit()

#### Fetch the tables

In [7]:
cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
""")

for table in cursor.fetchall():
    print(table[0])

Authors
BookAuthors
Books
Customers
OrderItems
Orders


#### Populate the tables with Data

In [8]:
authors_data = [
    (1, 'Daniel Carter'),
    (2, 'Sophia Bennett'),
    (3, 'James Mitchell'),
    (4, 'Olivia Parker'),
    (5, 'Ethan Brooks'),
    (6, 'Isabella Morgan'),
    (7, 'Noah Williams'),
    (8, 'Amelia Foster')
]

books_data = [
    (101, 'The Art of Algorithms', 2021, 59900, 25, 9781000000001),
    (102, 'Python Beyond Basics', 2023, 74900, 18, 9781000000002),
    (103, 'Foundation of Databases', 2020, 54900, 30, 9781000000003),
    (104, 'The Quantum Frontier', 2024, 89900, 12, 9781000000004),
    (105, 'Data Science Handbook', 2022, 69900, 22, 9781000000005),
    (106, 'Neural Network Explained', 2024, 79900, 15, 9781000000006),
    (107, 'The Silent Observer', 2019, 39900, 40, 9781000000007),
    (108, 'Beyond the Stars', 2021, 44900, 35, 9781000000008),
    (109, 'Modern SQL Mastery', 2025, 64900, 20, 9781000000009),
    (110, 'Intelligence Unbound', 2025, 99900, 10, 9781000000010)
]

bookauthor_data = [
    (1, 101),
    (1, 106),
    (2, 102),
    (2, 105),
    (3, 103),
    (3, 109),
    (4, 104),
    (4, 110),
    (5, 106),
    (5, 105),
    (6, 107),
    (6, 108),
    (7, 108)   
]

customer_data = [
    (201, 'Arjun Sharma', 'arjun@example.com', '+919876500001'),
    (202, 'Priya Verma', 'priya@example.com', '+919876500002'),
    (203, 'Rahul Singh', 'rahul@example.com', '+919876500003'),
    (204, 'Aisha Khan', 'aisha@example.com', '+919876500004'),
    (205, 'Rohan Mehta', 'rohan@example.com', '+919876500005'),
    (206, 'Neha Gupta', 'neha@example.com', '+919876500006'),
    (207, 'Kabir Malhotra', 'kabir@example.com', '+919876500007'),
    (208, 'Sara Thomas', 'sara@example.com', '+919876500008')
]

orders_data = [
    (5001, 201, '2026-01-05T10:30:00Z', 'completed'),
    (5002, 202, '2026-01-08T12:15:00Z', 'completed'),
    (5003, 203, '2026-01-12T15:45:00Z', 'completed'),
    (5004, 201, '2026-01-18T09:20:00Z', 'completed'),
    (5005, 204, '2026-02-02T11:00:00Z', 'pending'),
    (5006, 205, '2026-02-05T14:30:00Z', 'completed'),
    (5007, 206, '2026-02-10T16:00:00Z', 'cancelled'),
    (5008, 202, '2026-02-14T10:45:00Z', 'completed'),
    (5009, 207, '2026-02-20T13:10:00Z', 'pending'),
    (5010, 208, '2026-03-01T08:30:00Z', 'completed'),
    (5011, 203, '2026-03-05T17:20:00Z', 'completed'),
    (5012, 205, '2026-03-10T12:00:00Z', 'pending')
]

orderitems_data = [
    (1, 5001, 101, 1, 59900),
    (2, 5001, 102, 2, 74900),
    (3, 5002, 103, 1, 54900),
    (4, 5002, 105, 1, 69900),
    (5, 5003, 104, 1, 89900),
    (6, 5003, 106, 2, 79900),
    (7, 5004, 107, 3, 39900),
    (8, 5004, 108, 1, 44900),
    (9, 5005, 109, 2, 64900),
    (10, 5005, 110, 1, 99900),
    (11, 5006, 101, 2, 59900),
    (12, 5006, 103, 1, 54900),
    (13, 5007, 105, 1, 69900),
    (14, 5008, 102, 1, 74900),
    (15, 5008, 109, 1, 64900),
    (16, 5009, 106, 1, 79900),
    (17, 5010, 108, 2, 44900),
    (18, 5010, 110, 1, 99900),
    (19, 5011, 104, 1, 89900),
    (20, 5012, 107, 2, 39900)
]

conn.execute('PRAGMA foreign_keys = ON;')

try:
    conn.execute("BEGIN")

    cursor.executemany(
        "INSERT INTO Authors (author_id, name) VALUES (?, ?)",
        authors_data
    )

    cursor.executemany(
        """INSERT INTO Books
        (book_id, title, publication_year, price_paise, stock_quantity, isbn)
        VALUES (?, ?, ?, ?, ?, ?)""",
        books_data
    )

    cursor.executemany(
        "INSERT INTO BookAuthors (author_id, book_id) VALUES (?, ?)",
        bookauthor_data
    )

    cursor.executemany(
        """INSERT INTO Customers
        (customer_id, name, email, phone)
        VALUES (?, ?, ?, ?)""",
        customer_data
    )

    cursor.executemany(
        """INSERT INTO Orders
        (order_id, customer_id, order_date, status)
        VALUES (?, ?, ?, ?)""",
        orders_data
    )

    cursor.executemany(
        """INSERT INTO OrderItems
        (order_item_id, order_id, book_id, quantity, unit_price)
        VALUES (?, ?, ?, ?, ?)""",
        orderitems_data
    )

    conn.commit()
    print('All records inserted successfully')

except sqlite3.Error as e:
    conn.rollback()
    print("Database Error: ", e)


All records inserted successfully


#### Check if the populating process was successful and fetching records in each tables

In [2]:
tables = [
    "Authors",
    "Books",
    "BookAuthors",
    "Customers",
    "Orders",
    "OrderItems"
]

for table in tables:
    count = conn.execute(
        f'SELECT COUNT(*) FROM {table}'
    ).fetchone()[0]

    print(f'{table}: {count} records')

Authors: 8 records
Books: 10 records
BookAuthors: 13 records
Customers: 8 records
Orders: 12 records
OrderItems: 20 records


#### Check for foreign-key integrity

In [122]:
cursor.execute("PRAGMA foreign_key_check;")
violations = cursor.fetchall()

if violations:
    print("Foreign-key violations found!")
    print(violations)
else:
    print("Database passed foreign-key integrity check!")

Database passed foreign-key integrity check!


#### Retrieving table names and their columns

In [ ]:
tables = cursor.execute(
    'SELECT name FROM sqlite_master WHERE type="table"'
).fetchall()

for table in tables:
    table_name = table[0]

    columns = cursor.execute(
        f'PRAGMA table_info("{table_name}")'
    ).fetchall()

    print(f'\nTable: {table_name}')

    for column in columns:
        print(
            f"Column: {column[1]}, "
            f"Type: {column[2]}, "
            f"Not Null: {column[3]}, "
            f"Primary Key: {column[5]}"
        )

#### Testing UNIQUE constraints enforced via composite primary id

In [28]:
try:
    cursor.execute('INSERT INTO BookAuthors(author_id, book_id) VALUES(1, 101)').fetchall()
except sqlite3.Error as e:
    print(f"Error: {e}")

conn.rollback()

print(conn.in_transaction)

Error: UNIQUE constraint failed: BookAuthors.author_id, BookAuthors.book_id
False


#### Testing duplicate value insertion contraint on Books(isbn)


In [4]:
try:
    cursor.execute("INSERT INTO Books(book_id, title, publication_year, price_paise, stock_quantity, isbn)" \
    "VALUES(111, 'Here and after', 2026, 59900, 10, 9781000000001)")
    print("Unexpected: insertion succeeded")

except sqlite3.IntegrityError as e:
    print(f"Constraint enforced: {e}")

Constraint enforced: UNIQUE constraint failed: Books.isbn


#### Testing negative constraint in stock on Books(quantity)

In [5]:
try:
    cursor.execute("INSERT INTO Books(book_id, title, publication_year, price_paise, stock_quantity, isbn)" \
    "VALUES(111, 'Here and after', 2026, 59900, -10, 9781000000001)")
    print("Unexpected: insertion succeeded")

except sqlite3.IntegrityError as e:
    print(f"Constraint enforced: {e}")

Constraint enforced: CHECK constraint failed: stock_quantity >= 0


#### Testing zero quantity constraint on Books(quantity)

In [8]:
try:
    cursor.execute("""
        INSERT INTO OrderItems
        (order_item_id, order_id, book_id, quantity, unit_price)
        VALUES (21, 5001, 108, 0, 59900)
    """)

except sqlite3.IntegrityError as e:
    print(f"Constraint enforced: {e}")

Constraint enforced: CHECK constraint failed: quantity > 0


#### Testing invalid foreign key constraint by insertion in Orders(customer_id)

In [39]:
try:
    cursor.execute("""
        INSERT INTO Orders
        (order_id, customer_id, order_date, status)
        VALUES(5013, 999, '2026-03-10T12:00:00Z', 'completed')
""")

except sqlite3.IntegrityError as e:
    print(f'Constraint enforced: {e}')

Constraint enforced: FOREIGN KEY constraint failed


In [38]:
conn.rollback()
print(conn.in_transaction)

False


#### Basic SQL queries

In [57]:
# Books published after 2022
books_2022 = run_query("SELECT title, publication_year FROM Books WHERE publication_year > 2022")
display(books_2022)

# Books costing more than $600
price_600 = run_query("SELECT book_id, title, price_paise/100.0 FROM Books WHERE price_paise > 60000")
display(price_600)

# All books sorted by stock, highest first
max_stock = run_query("SELECT book_id, title, stock_quantity FROM Books ORDER BY stock_quantity DESC")
display(max_stock)

# Most expensive book
max_price = run_query("SELECT title, price_paise/100.0 FROM Books ORDER BY price_paise DESC LIMIT 1")
display(max_price)

# Books published between 2020 and 2024
btw_22_24 = run_query("SELECT title, publication_year FROM Books WHERE publication_year BETWEEN 2020 AND 2024 ORDER BY publication_year")
display(btw_22_24)

,title,publication_year
0,Python Beyond Basics,2023
1,The Quantum Frontier,2024
2,Neural Network Explained,2024
3,Modern SQL Mastery,2025
4,Intelligence Unbound,2025


,book_id,title,price_paise/100.0
0,102,Python Beyond Basics,749.0
1,104,The Quantum Frontier,899.0
2,105,Data Science Handbook,699.0
3,106,Neural Network Explained,799.0
4,109,Modern SQL Mastery,649.0
5,110,Intelligence Unbound,999.0


,book_id,title,stock_quantity
0,107,The Silent Observer,40
1,108,Beyond the Stars,35
2,103,Foundation of Databases,30
3,101,The Art of Algorithms,25
4,105,Data Science Handbook,22
5,109,Modern SQL Mastery,20
6,102,Python Beyond Basics,18
7,106,Neural Network Explained,15
8,104,The Quantum Frontier,12
9,110,Intelligence Unbound,10


,title,price_paise/100.0
0,Intelligence Unbound,999.0


,title,publication_year
0,Foundation of Databases,2020
1,The Art of Algorithms,2021
2,Beyond the Stars,2021
3,Data Science Handbook,2022
4,Python Beyond Basics,2023
5,The Quantum Frontier,2024
6,Neural Network Explained,2024


#### JOIN Operations

In [56]:
#Display every book alongside its authors
book_authors = run_query("""
    SELECT b.title, a.name
    FROM Books AS b
    LEFT JOIN BookAuthors AS ba
        ON b.book_id = ba.book_id
    LEFT JOIN Authors AS a
        ON a.author_id = ba.author_id
""")
display(book_authors)

#Display every order alingside the customer's name
customer_orders = run_query("""
    SELECT c.name, o.order_id FROM Customers AS c
    LEFT JOIN Orders AS o
        ON c.customer_id = o.customer_id
""")
display(customer_orders)

#Display order ID, customer name, book title, quantity and unit price
order_summary = run_query("""
    SELECT o.order_id, c.name, b.title, oi.quantity, oi.unit_price/100.0
    FROM Orders AS o
    LEFT JOIN Customers AS c
        ON o.customer_id = c.customer_id
    LEFT JOIN OrderItems AS oi
        ON o.order_id = oi.order_id
    LEFT JOIN Books AS b
        ON oi.book_id = b.book_id
    ORDER BY o.order_id
""")
display(order_summary)

#Authors with no books assigned
no_book = run_query("""
    SELECT a.name
    FROM Authors AS a
    LEFT JOIN BookAuthors AS ba
        ON a.author_id = ba.author_id
    WHERE ba.author_id IS NULL;
""")
display(no_book)

,title,name
0,The Art of Algorithms,Daniel Carter
1,Python Beyond Basics,Sophia Bennett
2,Foundation of Databases,James Mitchell
3,The Quantum Frontier,Olivia Parker
4,Data Science Handbook,Sophia Bennett
5,Data Science Handbook,Ethan Brooks
6,Neural Network Explained,Daniel Carter
7,Neural Network Explained,Ethan Brooks
8,The Silent Observer,Isabella Morgan
9,Beyond the Stars,Isabella Morgan


,name,order_id
0,Arjun Sharma,5001
1,Arjun Sharma,5004
2,Priya Verma,5002
3,Priya Verma,5008
4,Rahul Singh,5003
5,Rahul Singh,5011
6,Aisha Khan,5005
7,Rohan Mehta,5006
8,Rohan Mehta,5012
9,Neha Gupta,5007


,order_id,name,title,quantity,oi.unit_price/100.0
0,5001,Arjun Sharma,The Art of Algorithms,1,599.0
1,5001,Arjun Sharma,Python Beyond Basics,2,749.0
2,5002,Priya Verma,Foundation of Databases,1,549.0
3,5002,Priya Verma,Data Science Handbook,1,699.0
4,5003,Rahul Singh,The Quantum Frontier,1,899.0
5,5003,Rahul Singh,Neural Network Explained,2,799.0
6,5004,Arjun Sharma,The Silent Observer,3,399.0
7,5004,Arjun Sharma,Beyond the Stars,1,449.0
8,5005,Aisha Khan,Modern SQL Mastery,2,649.0
9,5005,Aisha Khan,Intelligence Unbound,1,999.0


,name
0,Amelia Foster


#### SQL Aggregation

In [54]:
# Calculate the total number of books in the inventory
df = run_query("""
    SELECT SUM(stock_quantity) AS total_stock FROM Books;
""")
display(df)

# Calculate the average selling price for each book
df = run_query("""
    SELECT b.title, AVG(oi.unit_price/100.0) AS avg_price
    FROM Books AS b
    LEFT JOIN OrderItems AS oi
        ON b.book_id = oi.book_id
    LEFT JOIN Orders AS o
        ON oi.order_id = o.order_id
    WHERE o.status = 'completed'
    GROUP BY b.book_id, b.title
    ORDER BY AVG(oi.unit_price)
""")
display(df)

# Calculate total copies sold for each book
df = run_query("""
    SELECT b.title, SUM(oi.quantity) AS total_sale
    FROM OrderItems AS oi
    JOIN Books AS b
        ON oi.book_id = b.book_id
    JOIN Orders AS o
        ON oi.order_id = o.order_id
    WHERE o.status = 'completed'
    GROUP BY b.title, oi.book_id
    ORDER BY SUM(oi.quantity)
""")
display(df)


# Calculate the total number of orders placed by each customer
df = run_query("""
    SELECT c.name, COUNT(o.order_id) AS total_order
    FROM Customers AS c
    LEFT JOIN Orders AS o
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.name
    ORDER BY COUNT(o.order_id)
""")
display(df)

,total_stock
0,227


,title,avg_price
0,The Silent Observer,399.0
1,Beyond the Stars,449.0
2,Foundation of Databases,549.0
3,The Art of Algorithms,599.0
4,Modern SQL Mastery,649.0
5,Data Science Handbook,699.0
6,Python Beyond Basics,749.0
7,Neural Network Explained,799.0
8,The Quantum Frontier,899.0
9,Intelligence Unbound,999.0


,title,total_sale
0,Data Science Handbook,1
1,Intelligence Unbound,1
2,Modern SQL Mastery,1
3,Foundation of Databases,2
4,Neural Network Explained,2
5,The Quantum Frontier,2
6,Beyond the Stars,3
7,Python Beyond Basics,3
8,The Art of Algorithms,3
9,The Silent Observer,3


,name,total_order
0,Aisha Khan,1
1,Neha Gupta,1
2,Kabir Malhotra,1
3,Sara Thomas,1
4,Arjun Sharma,2
5,Priya Verma,2
6,Rahul Singh,2
7,Rohan Mehta,2


### Business Analysis
#### 1. Sales and revenue analysis

In [53]:
#Q1- How much revenue has the bookstore generate from completed orders?
df = run_query("""
    SELECT SUM((oi.quantity * oi.unit_price)/100.0) AS total_amount
    FROM Orders AS o
    JOIN OrderItems AS oi
        ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
""")
display(df)

,total_amount
0,13429.0


In [52]:
#Which books generate the most revenue?
df = run_query("""
    SELECT b.title, SUM((oi.quantity * oi.unit_price)/100.0) AS total_amount, SUM(oi.quantity) AS copies_sold
    FROM OrderItems AS oi
    JOIN Books AS b
        ON oi.book_id = b.book_id
    JOIN Orders AS o
        ON oi.order_id = o.order_id
    WHERE o.status = 'completed'
    GROUP BY b.title, oi.book_id
    ORDER BY SUM(oi.quantity) DESC
""")
display(df)

,title,total_amount,copies_sold
0,Beyond the Stars,1347.0,3
1,Python Beyond Basics,2247.0,3
2,The Art of Algorithms,1797.0,3
3,The Silent Observer,1197.0,3
4,Foundation of Databases,1098.0,2
5,Neural Network Explained,1598.0,2
6,The Quantum Frontier,1798.0,2
7,Data Science Handbook,699.0,1
8,Intelligence Unbound,999.0,1
9,Modern SQL Mastery,649.0,1


In [51]:
#Which month generates the most revenue?
try:
    df = run_query("""
    SELECT strftime('%Y-%m', o.order_date) AS sales_month,
    SUM(oi.quantity * oi.unit_price)/100.0 AS monthly_revenue,
    COUNT(DISTINCT o.order_id) AS total_orders
    FROM Orders AS o
    JOIN OrderItems AS oi
        ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY sales_month
    ORDER BY sales_month
""")
except sqlite3.Error as e:
    print(f"Error: {e}")
display(df)

,sales_month,monthly_revenue,total_orders
0,2026-01,7488.0,4
1,2026-02,3145.0,2
2,2026-03,2796.0,2


In [47]:
# How much does a customer spend on average per completed order?
try:
    df = run_query("""
        SELECT AVG(order_total) AS average_order_value
        FROM (
            SELECT o.order_id,
                   SUM(oi.quantity * oi.unit_price) / 100.0 AS order_total
            FROM Orders AS o
            JOIN OrderItems AS oi
                ON o.order_id = oi.order_id
            WHERE o.status = 'completed'
            GROUP BY o.order_id
        )
    """)
except sqlite3.Error as e:
    print(f"Error: {e}")
display(df)


,average_order_value
0,1678.625


#### 2. Customer Analysis

In [41]:
#Which customer generates the most revenue?
try:
    df= run_query("""
    SELECT c.name AS Customer, COUNT(DISTINCT o.order_id) AS completed_orders, SUM(oi.quantity * oi.unit_price) / 100.0 AS spendings
    FROM Orders AS o
    JOIN Customers AS c
        ON o.customer_id = c.customer_id
    JOIN OrderItems AS oi
        ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY c.name
    ORDER BY SUM(oi.unit_price) DESC
""")
except sqlite3.Error as e:
    print(f"Error: {e}")
display(df)

,Customer,completed_orders,spendings
0,Priya Verma,2,2646.0
1,Rahul Singh,2,3396.0
2,Arjun Sharma,2,3743.0
3,Sara Thomas,1,1897.0
4,Rohan Mehta,1,1747.0


In [35]:
# Which customer has no completed order?
try:
    df = run_query("""
        SELECT c.name AS Customer
        FROM Customers AS c
        LEFT JOIN Orders AS o
            ON c.customer_id = o.customer_id
            AND o.status = 'completed'
        WHERE o.order_id IS NULL
    """)
except sqlite3.Error as e:
    print(f"Error: {e}")

display(df)

,Customer
0,Aisha Khan
1,Neha Gupta
2,Kabir Malhotra


#### 3. Inventory and product analysis

In [26]:
# Which books need stocking?
try:
    df = run_query("""
    SELECT title, stock_quantity, price_paise/100 AS price_in_rupee
    FROM Books
    WHERE stock_quantity < 15
    ORDER BY stock_quantity DESC
""")
except sqlite3.Error as e:
    print(f'Error: {e}')
display(df)

,title,stock_quantity,price_in_rupee
0,The Quantum Frontier,12,899
1,Intelligence Unbound,10,999


In [50]:
# What is the total moneytory value of current inventory?
try:
    df = run_query("""
    SELECT title,SUM(stock_quantity * price_paise) / 100.0 AS total_inventory_value
    FROM Books
    GROUP BY title
    ORDER BY SUM(stock_quantity * price_paise) / 100.0
""")
except sqlite3.Error as e:
    print(f'Error: {e}')
display(df)

,title,total_inventory_value
0,Intelligence Unbound,9990.0
1,The Quantum Frontier,10788.0
2,Neural Network Explained,11985.0
3,Modern SQL Mastery,12980.0
4,Python Beyond Basics,13482.0
5,The Art of Algorithms,14975.0
6,Data Science Handbook,15378.0
7,Beyond the Stars,15715.0
8,The Silent Observer,15960.0
9,Foundation of Databases,16470.0


In [42]:
#Which book sells the highest number of copies?
try:
    df = run_query("""
    SELECT
    b.book_id,
    b.title,
    COALESCE(
        SUM(
            CASE
                WHEN o.status = 'completed'
                THEN oi.quantity
                ELSE 0
            END
        ), 0
    ) AS total_copies_sold
FROM Books AS b
LEFT JOIN OrderItems AS oi
    ON b.book_id = oi.book_id
LEFT JOIN Orders AS o
    ON oi.order_id = o.order_id
GROUP BY b.book_id, b.title
ORDER BY total_copies_sold DESC;
""")
except sqlite3.Error as e:
    print(f'ErrorL {e}')
display(df)

,book_id,title,total_copies_sold
0,101,The Art of Algorithms,3
1,102,Python Beyond Basics,3
2,107,The Silent Observer,3
3,108,Beyond the Stars,3
4,103,Foundation of Databases,2
5,104,The Quantum Frontier,2
6,106,Neural Network Explained,2
7,105,Data Science Handbook,1
8,109,Modern SQL Mastery,1
9,110,Intelligence Unbound,1


#### Advanced analysis

In [49]:
#Which authors are associated with the highest revenue?
try:
    df = run_query("""
    SELECT a.name, SUM(oi.quantity * oi.unit_price)/100.0 AS author_revenue
    FROM OrderItems AS oi
    JOIN BookAuthors AS ba
        ON oi.book_id = ba.book_id
    JOIN Authors AS a
        ON ba.author_id = a.author_id
    JOIN Orders AS o
        ON oi.order_id = o.order_id
    WHERE o.status = 'completed'
    GROUP BY a.author_id, a.name
    ORDER BY author_revenue DESC
""")
except sqlite3.Error as e:
    print(f"Error: {e}")
display(df)

,name,author_revenue
0,Daniel Carter,3395.0
1,Sophia Bennett,2946.0
2,Olivia Parker,2797.0
3,Isabella Morgan,2544.0
4,Ethan Brooks,2297.0
5,James Mitchell,1747.0
6,Noah Williams,1347.0


In [48]:
#Are books being sold at prices different from their catalogue prices?
try:
    df = run_query("""
    SELECT b.book_id, b.title, (b.price_paise)/100.0 AS current_price, AVG((oi.unit_price)/100.0) AS avg_selling_price, ((b.price_paise/100.0) - AVG((oi.unit_price/100))) AS avg_price_difference
    FROM OrderItems AS oi
    JOIN Books AS b
        On oi.book_id = b.book_id
    JOIN Orders as o
        ON oi.order_id = o.order_id
    WHERE o.status = 'completed'
    GROUP BY b.book_id
    ORDER BY (b.price_paise)/100.0 - AVG((oi.unit_price)/100.0) DESC
""")
except sqlite3.Error as e:
    print(f"Error: {e}")
display(df)

,book_id,title,current_price,avg_selling_price,avg_price_difference
0,110,Intelligence Unbound,999.0,999.0,0.0
1,109,Modern SQL Mastery,649.0,649.0,0.0
2,108,Beyond the Stars,449.0,449.0,0.0
3,107,The Silent Observer,399.0,399.0,0.0
4,106,Neural Network Explained,799.0,799.0,0.0
5,105,Data Science Handbook,699.0,699.0,0.0
6,104,The Quantum Frontier,899.0,899.0,0.0
7,103,Foundation of Databases,549.0,549.0,0.0
8,102,Python Beyond Basics,749.0,749.0,0.0
9,101,The Art of Algorithms,599.0,599.0,0.0


# Findings and Conclusion
### Findings from the saved query outputs
- The notebook created six related tables and inserted sample rows: 8 authors, 10 books, 13 book-author links, 8 customers, 12 orders, and 20 order items. The recorded foreign-key check found no violations, and the constraint examples rejected invalid records.
- The completed-order revenue query reports ₹13,429. The monthly query reports January 2026 as the highest of the displayed months at ₹7,488, followed by February at ₹3,145 and March at ₹2,796.
- The average completed-order value query reports ₹1,678.625. The inventory query totals 227 copies; two books fall below the chosen 15-copy threshold: The Quantum Frontier (12) and Intelligence Unbound (10).
- The saved stock query shows The Silent Observer with 5 copies sold, the highest number in that query result. That query does not restrict orders to completed status, so this is a result of the query as written, not a verified completed-sales ranking.

### Conclusion and limitations:
This project demonstrates a six-table relational design, foreign-key relationships, integrity constraints, Python database access, and a range of SQL joins and aggregations. The recorded figures describe only the notebook’s small sample dataset. Several business queries need correction before their outputs can be used as reliable conclusions: customer spending omits quantity, the no-completed-order filter is malformed, some sales queries include non-completed orders, and book revenue output does not aggregate copies sold correctly. The author revenue query also counts a co-authored book’s revenue for each author.

Therefore, the revenue and monthly figures above are reported as the notebook displays them, while customer, book-ranking, and author-ranking outputs should be treated as provisional until those queries are fixed and rerun. The sample data cannot support claims about real bookstore demand, profitability, or customer behavior.

# AUTHOR

##### **Mohammad Arshad**